# Energy-Aware Client Selection in Federated Learning
### Interactive Tutorial & Benchmarking Guide
**Authors:** Khagesh Ranjan & Sudipto Ghosh

This notebook demonstrates how to:
1. Formulate and generate synthetic federated learning client selection instances.
2. Simulate selection policies under energy and battery constraints.
3. Compare heuristic and optimization policies (`Random`, `Fastest-First`, `Myopic Greedy`, `Battery-Aware Greedy`).
4. Verify solution feasibility with the independent checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from flsel.instance import generate
from flsel.policies import simulate, POLICIES
from flsel.checker import check
from flsel.greedy import modified_greedy, coverage_value

print("flsel package loaded successfully!")

## 1. Generating a Synthetic Problem Instance

We create a realistic edge environment with $K=40$ clients, $G=20$ data classes/groups, and $R=10$ communication rounds.

In [ ]:
inst = generate(K=40, G=20, R=10, seed=42)

print(f"Clients (K): {inst.K}")
print(f"Data Groups (G): {inst.G}")
print(f"Rounds (R): {inst.R}")
print(f"Deadline-eligible clients: {inst.elig.sum()} / {inst.K}")
print(f"Mean client energy per participation: {inst.e.mean():.2f}")
print(f"Mean round energy budget: {inst.E.mean():.2f}")

## 2. Comparing Selection Policies

We evaluate four client selection policies across all rounds:
- **Random**: Uniformly picks eligible clients until budget/cardinality is filled.
- **Fastest-First**: Prioritizes clients with the lowest round latency $\tau_k$.
- **Myopic Greedy**: Maximizes single-round marginal coverage gain without considering future battery depletion.
- **Battery-Aware Greedy**: Discounts clients with depleted batteries via effective cost $e_k (1 + \gamma \cdot \text{depletion}_k)$.

In [ ]:
policies = ["random", "fastest", "myopic", "battery_aware"]
results = {}

for pol in policies:
    X, met = simulate(inst, policy=pol, seed=42)
    feasible, obj, _ = check(inst, X)
    results[pol] = {"X": X, "metrics": met, "feasible": feasible, "obj": obj}
    print(f"{pol:15s} | Feasible: {feasible} | Coverage: {met['cov_pct']:.2f}% | Exhausted Clients: {met['exhausted_clients']} | Jain: {met['jain']:.4f}")

## 3. Visualizing Performance Metrics

In [ ]:
pol_names = list(results.keys())
cov_vals = [results[p]["metrics"]["cov_pct"] for p in pol_names]
exh_vals = [results[p]["metrics"]["exhausted_clients"] for p in pol_names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(pol_names, cov_vals, color=['#999999', '#7777aa', '#33aa33', '#1166dd'])
ax1.set_title("Weighted Group Coverage (%)")
ax1.set_ylabel("Coverage (%)")
ax1.set_ylim(0, 100)
for i, v in enumerate(cov_vals):
    ax1.text(i, v + 2, f"{v:.1f}%", ha='center', fontweight='bold')

ax2.bar(pol_names, exh_vals, color=['#e66101', '#fdb863', '#b2abd2', '#5e3c99'])
ax2.set_title("Exhausted Clients (Lower is Better)")
ax2.set_ylabel("Number of Devices")
for i, v in enumerate(exh_vals):
    ax2.text(i, v + 0.3, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Verification with the Independent Checker

In [ ]:
X_opt = results["battery_aware"]["X"]
is_feasible, total_cov, violations = check(inst, X_opt)

print(f"Verification status: {'FEASIBLE' if is_feasible else 'VIOLATIONS DETECTED'}")
print(f"Total objective: {total_cov}")
if violations:
    print("Violations:", violations)